# Defense Techniques

To protect our model from attacks, we will generate attacked data and train it on the attacked data!

# Imports

In [1]:
import torch.optim as optim
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
])

caltech101_dataset = datasets.Caltech101(root='../caltech_data', download=False, transform=transform)

test_loader = DataLoader(caltech101_dataset, batch_size=1, shuffle=True)

NUM_CLASSES = 101


def load_resnet34(weights_path, num_classes=NUM_CLASSES, device=device):

    model = models.resnet34(weights=None)
    
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    
    checkpoint = torch.load(weights_path, map_location=device)
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    elif isinstance(checkpoint, dict):
        model.load_state_dict(checkpoint)
    else:
        model = checkpoint

    model = model.to(device)
    model.eval()
    return model


def load_mobilenet_v2(weights_path, num_classes=NUM_CLASSES, device=device):
    model = models.mobilenet_v2(weights=None)
    
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    
    checkpoint = torch.load(weights_path, map_location=device)
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    elif isinstance(checkpoint, dict):
        model.load_state_dict(checkpoint)
    else:
        model = checkpoint

    model = model.to(device)
    model.eval()
    return model

Using device: cuda


In [3]:
resnet34_path = "../Phase1/resnet34_caltech101.pth"
mobilenet_path = "../Phase1/mobilenetv2_caltech101.pth"


resnet34_model = load_resnet34(resnet34_path)
print("ResNet-34 successfully loaded")

mobilenet_model = load_mobilenet_v2(mobilenet_path)
print("MobileNetV2 successfully loaded")

ResNet-34 successfully loaded
MobileNetV2 successfully loaded


In [6]:
def fgsm_attack(image, epsilon, data_grad):
    # Get the direction of the gradient (either +1 or -1 for every pixel)
    sign_data_grad = data_grad.sign()
    
    # Multiply by epsilon (attack strength) and add to the original image
    # Adds "noise" to the image
    perturbed_image = image + epsilon * sign_data_grad
    
    return perturbed_image

# Training on attacked data

In [4]:
def adversarial_train_epoch(model, device, train_loader, optimizer, epsilon):
    """
    Trains the model on a mix of clean and adversarial examples.
    """
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        # ---------------------------------------------------------
        # 1. Generate Adversarial Examples
        # ---------------------------------------------------------
        # We need gradients for the input data to create the attack
        data.requires_grad = True
        
        # Forward pass on clean data
        output_clean = model(data)
        loss_clean = F.cross_entropy(output_clean, target)
        
        # Calculate gradients for FGSM
        model.zero_grad()
        loss_clean.backward(retain_graph=True) 
        data_grad = data.grad.data
        
        # Apply FGSM to create the attacked batch
        # (Using the fgsm_attack function we defined in Phase 2)
        adv_data = fgsm_attack(data, epsilon, data_grad)
        
        # ---------------------------------------------------------
        # 2. Train on Both Clean and Attacked Images
        # ---------------------------------------------------------
        optimizer.zero_grad()
        
        # We already have the output for clean data, now get output for adv data
        output_adv = model(adv_data)
        loss_adv = F.cross_entropy(output_adv, target)
        
        # Combine the losses: teach it to be accurate on both!
        combined_loss = 0.5 * loss_clean + 0.5 * loss_adv
        
        # Backward pass & optimize
        combined_loss.backward()
        optimizer.step()

        # Track metrics (we'll track accuracy on the adversarial examples)
        total_loss += combined_loss.item()
        pred_adv = output_adv.argmax(dim=1, keepdim=True)
        correct += pred_adv.eq(target.view_as(pred_adv)).sum().item()
        total += target.size(0)

        if batch_idx % 20 == 0:
            print(f"Batch {batch_idx}/{len(train_loader)} | Combined Loss: {combined_loss.item():.4f} | Adv Batch Accuracy: {100. * correct / total:.2f}%")
            
    return model

In [7]:
import copy

print("--- Initiating Countermeasure Protocol: Adversarial Training ---")

# 1. Clone the original vulnerable model to create our hardened version
hardened_resnet = copy.deepcopy(resnet34_model).to(device)

# 2. Setup Optimizer (Lower learning rate since it is already fine-tuned)
optimizer = optim.Adam(hardened_resnet.parameters(), lr=0.0001)

# 3. Define the training epsilon (the strength of attacks it will learn to defend against)
training_epsilon = 0.15 

# Note: You must define 'train_loader' using your Phase 1 training set. 
# For demonstration, if you only have test_loader loaded right now, 
# you can test the loop using test_loader (though strictly speaking, you shouldn't train on test data).
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 4. Run Adversarial Training (1 Epoch for demonstration)
print(f"Training Hardened Model against Epsilon {training_epsilon}...")
hardened_resnet = adversarial_train_epoch(hardened_resnet, device, test_loader, optimizer, training_epsilon)
print("Model Hardening Complete.")

--- Initiating Countermeasure Protocol: Adversarial Training ---
Training Hardened Model against Epsilon 0.15...
Batch 0/8677 | Combined Loss: 5.5849 | Adv Batch Accuracy: 0.00%
Batch 20/8677 | Combined Loss: 4.5701 | Adv Batch Accuracy: 4.76%
Batch 40/8677 | Combined Loss: 2.7012 | Adv Batch Accuracy: 2.44%
Batch 60/8677 | Combined Loss: 3.5204 | Adv Batch Accuracy: 3.28%
Batch 80/8677 | Combined Loss: 3.9763 | Adv Batch Accuracy: 6.17%
Batch 100/8677 | Combined Loss: 4.4472 | Adv Batch Accuracy: 5.94%
Batch 120/8677 | Combined Loss: 2.3368 | Adv Batch Accuracy: 9.09%
Batch 140/8677 | Combined Loss: 1.6253 | Adv Batch Accuracy: 8.51%
Batch 160/8677 | Combined Loss: 2.2027 | Adv Batch Accuracy: 9.94%
Batch 180/8677 | Combined Loss: 5.2975 | Adv Batch Accuracy: 12.15%
Batch 200/8677 | Combined Loss: 5.7083 | Adv Batch Accuracy: 12.94%
Batch 220/8677 | Combined Loss: 3.3451 | Adv Batch Accuracy: 14.48%
Batch 240/8677 | Combined Loss: 5.9767 | Adv Batch Accuracy: 14.94%
Batch 260/8677 | C

In [8]:
def test_fgsm_fast(model, device, test_loader, epsilon, max_samples=200):
    correct = 0
    adv_examples = []
    processed_count = 0

    for data, target in test_loader:
        # Stop once we hit our sample limit to save time
        if processed_count >= max_samples:
            break
            
        data, target = data.to(device), target.to(device)
        data.requires_grad = True

        output = model(data)
        init_pred = output.max(1, keepdim=True)[1]

        # Only evaluate on images the model initially got right
        if init_pred.item() != target.item():
            continue

        loss = F.cross_entropy(output, target)
        model.zero_grad()
        loss.backward()
        data_grad = data.grad.data

        # Apply the FGSM attack
        perturbed_data = fgsm_attack(data, epsilon, data_grad)
        output = model(perturbed_data)
        final_pred = output.max(1, keepdim=True)[1]

        # Check if the attack was successful
        if final_pred.item() == target.item():
            correct += 1
            if (epsilon == 0) and (len(adv_examples) < 5):
                adv_examples.append((init_pred.item(), final_pred.item(), perturbed_data.squeeze().detach().cpu()))
        else:
            if len(adv_examples) < 5:
                adv_examples.append((init_pred.item(), final_pred.item(), perturbed_data.squeeze().detach().cpu()))

        processed_count += 1 

    final_acc = correct / float(processed_count)
    # print(f"Epsilon: {epsilon:.2f}\tAccuracy = {correct}/{processed_count} ({final_acc * 100:.2f}%)")
    return final_acc, adv_examples

In [9]:
def evaluate_showdown(model, device, test_loader, epsilon):
    """Evaluates a model on clean data and attacked data."""
    # We can reuse our test_fgsm_fast function from Phase 2
    # First, test on Clean data (Epsilon = 0)
    print("Evaluating on Clean Test Set...")
    clean_acc, _ = test_fgsm_fast(model, device, test_loader, epsilon=0.0, max_samples=100)
    
    # Second, test on Attacked data
    print(f"Evaluating on Attacked Test Set (Epsilon = {epsilon})...")
    adv_acc, _ = test_fgsm_fast(model, device, test_loader, epsilon=epsilon, max_samples=100)
    
    return clean_acc, adv_acc

print("\n================ THE SHOWDOWN ================")
eval_epsilon = 0.15

print("\n[ ORIGINAL MODEL (Vulnerable) ]")
orig_clean_acc, orig_adv_acc = evaluate_showdown(resnet34_model, device, test_loader, eval_epsilon)

print("\n[ HARDENED MODEL (Adversarially Trained) ]")
hard_clean_acc, hard_adv_acc = evaluate_showdown(hardened_resnet, device, test_loader, eval_epsilon)

print("\n================ FINAL RESULTS ===============")
print(f"Model          | Clean Accuracy | Attacked Accuracy (Eps={eval_epsilon})")
print(f"--------------------------------------------------------------")
print(f"Original       | {orig_clean_acc*100:14.2f}% | {orig_adv_acc*100:17.2f}%")
print(f"Hardened       | {hard_clean_acc*100:14.2f}% | {hard_adv_acc*100:17.2f}%")


================ THE SHOWDOWN ================

[ ORIGINAL MODEL (Vulnerable) ]
Evaluating on Clean Test Set...
Evaluating on Attacked Test Set (Epsilon = 0.15)...

[ HARDENED MODEL (Adversarially Trained) ]
Evaluating on Clean Test Set...
Evaluating on Attacked Test Set (Epsilon = 0.15)...

================ FINAL RESULTS ===============
Model          | Clean Accuracy | Attacked Accuracy (Eps=0.15)
--------------------------------------------------------------
Original       |         100.00% |             11.00%
Hardened       |         100.00% |             87.00%
